# Patient 360

Builds the gold-layer `michelin.gold.patient_360` table from silver sources using the workbook mapping. This notebook performs a full rebuild each run and preserves all patients with left joins to hospital, insurance, and aggregated lab activity.

In [0]:
spark.sql("SELECT current_user()").show(truncate=False)

In [0]:
catalog_name = "michelin"
target_table = f"{catalog_name}.gold.patient_360"
source_tables = [
    (catalog_name, "silver", "patients"),
    (catalog_name, "silver", "hospitals"),
    (catalog_name, "silver", "insurance_providers"),
    (catalog_name, "silver", "lab_results"),
]


def table_is_listed(catalog_name: str, schema_name: str, table_name: str) -> bool:
    return (
        spark.sql(f"SHOW TABLES IN {catalog_name}.{schema_name} LIKE '{table_name}'").count() > 0
    )


missing_tables = [
    f"{catalog_name}.{schema_name}.{table_name}"
    for catalog_name, schema_name, table_name in source_tables
    if not table_is_listed(catalog_name, schema_name, table_name)
]

if missing_tables:
    print("Skipped build because these required source tables are not available yet:")
    for table_name in missing_tables:
        print(f" - {table_name}")
else:
    try:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.gold")
        spark.sql(
            f"""
            CREATE OR REPLACE TABLE {target_table} AS
            WITH patient_lab_agg AS (
              SELECT
                lr.patient_id,
                CAST(COUNT(*) AS INT) AS total_tests,
                CAST(SUM(CASE WHEN lr.result_status = 'Abnormal' THEN 1 ELSE 0 END) AS INT) AS abnormal_test_count,
                CAST(SUM(CASE WHEN lr.result_status = 'Critical' THEN 1 ELSE 0 END) AS INT) AS critical_test_count,
                MAX(CAST(lr.test_date AS DATE)) AS last_test_date
              FROM {catalog_name}.silver.lab_results AS lr
              GROUP BY lr.patient_id
            )
            SELECT
              p.patient_id,
              p.first_name,
              p.last_name,
              CAST(p.date_of_birth AS DATE) AS date_of_birth,
              CAST(FLOOR(DATEDIFF(CURRENT_DATE(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
              p.gender,
              p.city,
              p.state,
              p.registered_hospital_id AS hospital_id,
              h.hospital_name,
              p.insurance_provider_id,
              ip.provider_name AS insurance_provider_name,
              COALESCE(pla.total_tests, 0) AS total_tests,
              COALESCE(pla.abnormal_test_count, 0) AS abnormal_test_count,
              COALESCE(pla.critical_test_count, 0) AS critical_test_count,
              pla.last_test_date
            FROM {catalog_name}.silver.patients AS p
            LEFT JOIN {catalog_name}.silver.hospitals AS h
              ON p.registered_hospital_id = h.hospital_id
            LEFT JOIN {catalog_name}.silver.insurance_providers AS ip
              ON p.insurance_provider_id = ip.insurance_provider_id
            LEFT JOIN patient_lab_agg AS pla
              ON p.patient_id = pla.patient_id
            """
        )
        print(f"Created {target_table}")
    except Exception as exc:
        print(f"Build could not be completed for {target_table}.")
        print(str(exc))

In [0]:
target_table = "michelin.gold.patient_360"

if spark.catalog.tableExists(target_table):
    preview_df = spark.table(target_table).orderBy("patient_id")
    display(preview_df.limit(50))
    print(f"Total rows: {preview_df.count():,}")
else:
    print(f"Table not created yet: {target_table}")